In [ ]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch import optim

# Repo import
repo_dir = "/content/stk-mat2011"  # adjust if local
if repo_dir not in sys.path:
    sys.path.append(f"{repo_dir}/code/scripts")

from nnhmm import NNHMM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Choose your cleaned mega file
data_path = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed_mega/eurusd_mega_master_21_26.parquet")
df = pd.read_parquet(data_path)

# Expecting at least datetime + returns
df = df.dropna(subset=["returns"]).copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

# Optional robust clip to reduce extreme outliers
q_lo, q_hi = df["returns"].quantile([0.001, 0.999])
df["returns"] = df["returns"].clip(q_lo, q_hi)

print(df.shape, df["datetime"].min(), df["datetime"].max())
df.head()

In [ ]:
# Time-based split
n = len(df)
i1 = int(0.70 * n)
i2 = int(0.85 * n)

df_tr = df.iloc[:i1].copy()
df_va = df.iloc[i1:i2].copy()
df_te = df.iloc[i2:].copy()

print("train:", len(df_tr), "val:", len(df_va), "test:", len(df_te))

In [ ]:
def to_tensor(x):
    return torch.tensor(x.values, dtype=torch.float32).unsqueeze(1).to(device)

y_tr = to_tensor(df_tr["returns"])
y_va = to_tensor(df_va["returns"])
y_te = to_tensor(df_te["returns"])

print(y_tr.shape, y_va.shape, y_te.shape)

In [ ]:
K = 2
model = NNHMM(n_states=K, input_dim=1).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)

EPOCHS = 400
PATIENCE = 30  # early stopping on val NLL

In [ ]:
best_val = np.inf
best_state = None
bad = 0

hist_tr, hist_va, hist_lr = [], [], []

for ep in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    ll_tr = model(y_tr)                 # total log-lik
    loss_tr = -ll_tr / len(y_tr)        # per sample NLL
    loss_tr.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        ll_va = model(y_va)
        loss_va = -ll_va / len(y_va)

    scheduler.step(loss_va)

    tr = float(loss_tr.item())
    va = float(loss_va.item())
    lr = optimizer.param_groups[0]["lr"]

    hist_tr.append(tr)
    hist_va.append(va)
    hist_lr.append(lr)

    if va < best_val - 1e-6:
        best_val = va
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1

    if ep % 25 == 0:
        print(f"ep={ep:04d} train_nll={tr:.6f} val_nll={va:.6f} lr={lr:.2e}")

    if bad >= PATIENCE:
        print(f"early stop at ep={ep}, best_val={best_val:.6f}")
        break

# restore best
model.load_state_dict(best_state)
print("best val nll:", best_val)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(hist_tr, label="train_nll")
plt.plot(hist_va, label="val_nll")
plt.title("NNHMM training")
plt.xlabel("epoch")
plt.ylabel("NLL per sample")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10,3))
plt.plot(hist_lr)
plt.title("learning rate")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
with torch.no_grad():
    trans = model.transition_matrix.detach().cpu().numpy()
    pi = model.pi.detach().cpu().numpy()
    mu = model.means.detach().cpu().numpy().ravel()
    sig = torch.exp(model.log_std).detach().cpu().numpy().ravel()

print("pi:", pi)
print("transition:\n", trans)
for k in range(K):
    print(f"state {k}: mean={mu[k]:.6e} std={sig[k]:.6e}")

In [ ]:
# Uncomment once predict_states is implemented in NNHMM
# with torch.no_grad():
#     z_te = model.predict_states(y_te).detach().cpu().numpy()
# 
# plt.figure(figsize=(14,4))
# plt.plot(df_te["datetime"].values, df_te["returns"].values, lw=0.6, color="black")
# plt.scatter(
#     df_te["datetime"].values,
#     df_te["returns"].values,
#     c=z_te,
#     s=4,
#     cmap="tab10",
# )
# plt.title("test returns with decoded states")
# plt.grid(alpha=0.3)
# plt.show()

In [ ]:
# Simple AR(1): r_t = a + b r_{t-1} + e_t
r_tr = df_tr["returns"].values
X = np.column_stack([np.ones(len(r_tr)-1), r_tr[:-1]])
y = r_tr[1:]
coef = np.linalg.lstsq(X, y, rcond=None)[0]
a_hat, b_hat = coef

# residual std on train
res = y - (a_hat + b_hat * r_tr[:-1])
s_hat = res.std(ddof=1)

print("AR(1) a,b,s:", a_hat, b_hat, s_hat)

# one-step predictions on test
r_te = df_te["returns"].values
pred = a_hat + b_hat * r_te[:-1]
true = r_te[1:]

rmse_ar1 = np.sqrt(np.mean((true - pred)**2))
lo = pred - 1.96 * s_hat
hi = pred + 1.96 * s_hat
cov95_ar1 = np.mean((true >= lo) & (true <= hi))

print("AR(1) test rmse:", rmse_ar1)
print("AR(1) 95% coverage:", cov95_ar1)

In [ ]:
model.eval()
with torch.no_grad():
    nll_te = float((-model(y_te) / len(y_te)).item())

print("NNHMM test NLL per sample:", nll_te)

In [ ]:
model.eval()
with torch.no_grad():
    nll_te = float((-model(y_te) / len(y_te)).item())

print("NNHMM test NLL per sample:", nll_te)

In [ ]:
run = {
    "seed": SEED,
    "K": K,
    "n_train": len(df_tr),
    "n_val": len(df_va),
    "n_test": len(df_te),
    "best_val_nll": float(best_val),
    "test_nll": float(nll_te),
    "ar1_test_rmse": float(rmse_ar1),
    "ar1_cov95": float(cov95_ar1),
}

summary = pd.DataFrame([run])
summary